# Hebrew Sentence Generator — DictaLM 2.0
Generates synthetic Hebrew sentences using `dicta-il/dictalm2.0` with 4-bit quantization.  
**Target**: 1 000 sentences for testing + time extrapolation to 1 000 000.

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q transformers accelerate bitsandbytes huggingface_hub

In [ ]:
# ── 2. GPU sanity check — STOP here if no GPU found ──────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ No GPU detected!\n"
        "Go to: Settings (right panel) → Accelerator → GPU T4 x2  →  Save\n"
        "Then restart the kernel and run again."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU detected: {gpu_name}  ({vram_gb:.1f} GB VRAM)")
print(f"   CUDA version : {torch.version.cuda}")

In [ ]:
# ── 3. Download English word dictionary (seed words) ─────────────────────────
import urllib.request
from pathlib import Path

DICT_URL  = "https://raw.githubusercontent.com/first20hours/google-10000-english/master/google-10000-english-no-swears.txt"
DICT_FILE = Path("/kaggle/working/words.txt")

if not DICT_FILE.exists():
    print("Downloading word dictionary...")
    tmp = Path("/kaggle/working/words_raw.txt")
    urllib.request.urlretrieve(DICT_URL, tmp)
    raw   = tmp.read_text(encoding="utf-8").splitlines()
    words = [
        w.strip().lower() for w in raw
        if 4 <= len(w.strip()) <= 12
        and w.strip().isalpha()
    ]
    tmp.unlink()
    DICT_FILE.write_text("\n".join(words), encoding="utf-8")
    print(f"✅ Saved {len(words):,} words → {DICT_FILE}")
else:
    count = sum(1 for _ in DICT_FILE.open())
    print(f"✅ words.txt already exists ({count:,} words)")

In [ ]:
# ── 4. Download model from HuggingFace ───────────────────────────────────────
from huggingface_hub import snapshot_download

MODEL_ID   = "dicta-il/dictalm2.0"
MODEL_PATH = "/kaggle/working/dictalm2"

if not Path(MODEL_PATH).exists():
    print(f"Downloading {MODEL_ID} (this can take a few minutes)...")
    snapshot_download(repo_id=MODEL_ID, local_dir=MODEL_PATH)
    print("✅ Model downloaded.")
else:
    print(f"✅ Model already present at {MODEL_PATH}")

In [ ]:
# ── 5. Load model + tokenizer (4-bit quantization) ───────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"Loading model from {MODEL_PATH} (4-bit)...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)
model.eval()

DEVICE = next(model.parameters()).device
HAS_CHAT_TEMPLATE = hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None

# Final device check — must be cuda
if DEVICE.type != "cuda":
    raise RuntimeError(f"❌ Model loaded on {DEVICE} instead of CUDA — check GPU settings!")

print(f"✅ Model ready. Device={DEVICE}")
print(f"   Chat template : {'yes ✓' if HAS_CHAT_TEMPLATE else 'no — plain Hebrew prefix'}")
print(f"   VRAM used     : {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── 6. Generation helpers ─────────────────────────────────────────────────────
import re
import random

WORDS = DICT_FILE.read_text(encoding="utf-8").splitlines()
WORDS = [w for w in WORDS if w.strip()]
print(f"Loaded {len(WORDS):,} seed words")

SYSTEM_PROMPT = (
    "אתה סופר יצירתי שכותב אך ורק בעברית. "
    "המשתמש יספק לך שתי מילים באנגלית — המשך מהן בצורה יצירתית וחופשית. "
    "צור המשך מקורי, מעניין ומפתיע בעברית, כאילו שתי המילים הן תחילת סיפור או תמונה. "
    "חל איסור מוחלט להשתמש באנגלית או בכל שפה אחרת. "
    "כתוב משפט עברי אחד בלבד."
)

MAX_NEW_TOKENS = 22
BATCH_SIZE     = 32
MIN_WORDS      = 3
MAX_WORDS      = 20
LATIN_RE       = re.compile(r"[A-Za-z]")

def random_seed() -> str:
    return " ".join(random.sample(WORDS, 2))

def build_prompt(seed: str) -> str:
    if HAS_CHAT_TEMPLATE:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"כתוב משפט אחד בעברית על הנושא: {seed}"},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        return f"{SYSTEM_PROMPT}\n\nכתוב משפט אחד בעברית על הנושא: {seed}\n"

def trim_to_sentence(text: str) -> str:
    if "\n" in text:
        text = text.split("\n")[0]
    for sep in [".", "!", "?"]:
        if sep in text:
            text = text.split(sep)[0].strip() + sep
            break
    text = text.strip()
    words = text.split()
    if len(words) > MAX_WORDS:
        text = " ".join(words[:MAX_WORDS])
        if text[-1] not in ".!?":
            text += "."
    return text

def is_valid(text: str) -> bool:
    """Not blank, not too short, no English, no broken UTF-8 sequences."""
    words = text.split()
    return len(words) >= MIN_WORDS and not LATIN_RE.search(text) and "�" not in text

def generate_batch(seeds: list) -> list:
    prompts = [build_prompt(s) for s in seeds]
    inputs  = tokenizer(
        prompts, return_tensors="pt", padding=True,
        truncation=True, max_length=128,
    ).to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=1.05,
            top_k=65,
            top_p=0.91,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.pad_token_id,
        )
    prompt_len = inputs["input_ids"].shape[1]
    return [trim_to_sentence(tokenizer.decode(o[prompt_len:], skip_special_tokens=True)) for o in outputs]

print("✅ Helpers ready")

In [ ]:
# ── 7. Generate 1 000 sentences ───────────────────────────────────────────────
import time
import datetime
from IPython.display import clear_output

N_SENTENCES = 1_000
TARGET      = 1_000_000
LOG_EVERY   = 5   # print progress every N batches

sentences       = []
batch_times     = []
total_generated = 0
skipped         = 0
batch_num       = 0
start_total     = time.perf_counter()

print(f"Generating {N_SENTENCES:,} Hebrew sentences (batch={BATCH_SIZE})...\n")

while len(sentences) < N_SENTENCES:
    batch_seeds = [random_seed() for _ in range(BATCH_SIZE)]

    t0      = time.perf_counter()
    results = generate_batch(batch_seeds)
    dt      = time.perf_counter() - t0
    batch_times.append(dt)
    total_generated += len(results)
    batch_num       += 1

    for r in results:
        if is_valid(r):
            sentences.append(r)
            if len(sentences) >= N_SENTENCES:
                break
        else:
            skipped += 1

    if batch_num % LOG_EVERY == 0 or len(sentences) >= N_SENTENCES:
        done     = min(len(sentences), N_SENTENCES)
        avg_sent = sum(batch_times) / len(batch_times) / BATCH_SIZE
        eta      = avg_sent * (N_SENTENCES - done)
        elapsed  = time.perf_counter() - start_total
        bar      = "█" * (done * 30 // N_SENTENCES) + "░" * (30 - done * 30 // N_SENTENCES)
        speed    = done / elapsed if elapsed > 0 else 0

        clear_output(wait=True)
        print(f"Generating {N_SENTENCES:,} Hebrew sentences (batch={BATCH_SIZE})...\n")
        print(f"  [{bar}] {done:>5}/{N_SENTENCES}")
        print(f"  Elapsed  : {elapsed:.0f}s")
        print(f"  Speed    : {speed:.2f} sent/sec  ({dt:.2f}s/batch)")
        print(f"  Skipped  : {skipped}")
        print(f"  ETA      : {eta:.0f}s")

total_time = time.perf_counter() - start_total
clear_output(wait=True)
print(f"✅ Done! Generated {len(sentences):,} sentences in {total_time:.1f}s")

In [ ]:
# ── 8. Save results + stats ───────────────────────────────────────────────────
ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = Path(f"/kaggle/working/generated_hebrew_{ts}.txt")

with out_path.open("w", encoding="utf-8") as f:
    for s in sentences:
        f.write(s + "\n")

file_size_kb = out_path.stat().st_size / 1024

throughput  = len(sentences) / total_time
ext_seconds = TARGET / throughput

print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Results  (batch={BATCH_SIZE}, template={'chat' if HAS_CHAT_TEMPLATE else 'prefix'})
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Valid sentences     : {len(sentences):,}
  Total generated     : {total_generated:,}
  Skipped (English)   : {skipped:,}
  Total wall time     : {total_time:.1f}s  ({total_time/60:.1f} min)
  Throughput          : {throughput:.2f} sent/sec

  Output file         : {out_path.name}
  File size           : {file_size_kb:.1f} KB

  ── Extrapolation to {TARGET:,} sentences ──
  Estimated time      : {ext_seconds:,.0f}s
    • {ext_seconds/3600:.1f} hours
    • {ext_seconds/86400:.1f} days
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print(f"💾 Saved → {out_path}")
print("\nPreview (first 5 sentences):")
for i, s in enumerate(sentences[:5], 1):
    print(f"  {i}. {s}")